# Starbucks store locator
Website: https://www.starbucks.co.uk/store-locator?types=starbucks&latLng=55.8625388%2C-4.284226000000002&zoom=12

Investigate the website and the feature of the URLs.

Get the latitudes, longitudes, addresses, Unique IDs, store names, and openning hours of all the Starbucks in the area below as a DataFrame.

Visualise the stores on map.

The boudanries to scrap:

north_bound = 60.8590 N

south_bound = 54.6356 N

west_bound = -7.385 W

east_bound = 1.7834 E

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import ast
from pandas import json_normalize
from tqdm import tqdm
import folium

In [2]:

import numpy as np
import time

north = 60.8590
south = 54.6356
west = -7.385
east = 1.7834

lat_step = 0.8
lng_step = 0.8

base_url = "https://www.starbucks.co.uk/api/v2/stores/"

stores = {}

for lat in np.arange(south, north, lat_step):
    for lng in np.arange(west, east, lng_step):

        params = {
            "filter[coordinates][latitude]": lat,
            "filter[coordinates][longitude]": lng,
            "filter[radius]": 15
        }

        response = requests.get(base_url, params=params)
        data = response.json()

        for store in data["data"]:

            sid = store["id"]
            attr = store["attributes"]

            if sid not in stores:

                stores[sid] = {
                    "id": sid,
                    "name": attr["name"],
                    "latitude": attr["coordinates"]["latitude"],
                    "longitude": attr["coordinates"]["longitude"],
                    "address": attr["address"]["streetAddressLine1"],
                    "city": attr["address"]["city"],
                    "postcode": attr["address"]["postalCode"],
                    "opening_hours": attr["openHours"]
                }

        time.sleep(0.3)

df = pd.DataFrame(stores.values())
df.head()

,id,name,latitude,longitude,address,city,postcode,opening_hours
0,1052389,Magherafelt - Meadowlane SC,54.75457,-6.61099,Unit 10 Meadowlane Shopping Centre Moneymore R...,Magherafelt,BT456PR,"[{'open': True, 'open24Hours': False, 'openTim..."
1,89721,Belfast Int Airport - Airside,54.66259,-6.21981,Belfast International Airport,Belfast,BT29 4AB,"[{'open': True, 'open24Hours': False, 'openTim..."
2,1051378,Belfast Holywood Exch Sainsbur,54.62640,-5.85683,302 Airport Road West Belfast BT3 9EJ,Belfast,BT3 9EJ,"[{'open': True, 'open24Hours': False, 'openTim..."
3,1009132,Bangor Springhill,54.65534,-5.69851,"Unit 7, Springhill Retail Park, Bangor, Co....",Bangor,BT19 1ND,"[{'open': True, 'open24Hours': False, 'openTim..."
4,1016117,Belfast City Airport - Airside,54.61580,-5.87304,Sydenham By Pass,Belfast,BT39JH,"[{'open': True, 'open24Hours': False, 'openTim..."


In [3]:

center_lat = df["latitude"].mean()
center_lng = df["longitude"].mean()

m = folium.Map(location=[center_lat, center_lng], zoom_start=7)

for _, row in df.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=f"{row['name']}<br>{row['address']}<br>{row['postcode']}"
    ).add_to(m)

m